# 00 — Setup & Verifikasi Lingkungan

**Tujuan:** memastikan lingkungan siap sebelum tahap apa pun (ingest, chunking, indexing, eksperimen).

**INPUT:** `../configs/base.yaml`, `.env` (GEMINI_API_KEY)

**OUTPUT:** `../results/environment.json`, seluruh direktori proyek dibuat (lihat `perancangan.md` §2)

**Perkiraan runtime:** 3-8 menit (mengunduh model embedding/reranker saat pertama kali dijalankan)

**Perkiraan biaya API:** ~1 panggilan Gemini API generasi pendek (< $0.001)

**Assert di notebook ini:** GPU CUDA terdeteksi (jika tersedia); model embedding & reranker berhasil dimuat dan melakukan satu forward pass; panggilan Gemini API mengembalikan respons non-kosong; seluruh direktori keluaran ada.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0,"../src")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import json
import platform
import sys as _sys
from pathlib import Path

from dotenv import load_dotenv

from config import Config, ensure_directories, load_config, set_seed

load_dotenv("../.env")
cfg = load_config("../configs/base.yaml")
set_seed(cfg.seed)
cfg

Config(condition='base', seed=42, paths=PathsConfig(data_raw_pdf='data/raw/pdf', data_raw_web='data/raw/web', data_interim='data/interim', data_processed='data/processed', data_benchmark='data/benchmark', indexes_faiss='indexes/faiss', indexes_bm25='indexes/bm25', runs='runs', results='results', figures='figures', cache_db='cache/llm_cache.sqlite'), models=ModelsConfig(embedding='BAAI/bge-m3', reranker='BAAI/bge-reranker-v2-m3', generator='gemini-3.6-flash', judge='gemini-3.6-flash'), chunking=ChunkingConfig(chunk_size=800, chunk_overlap=120, unit='char', min_chunk_chars=100), retrieval=RetrievalConfig(mode='hybrid', top_k_dense=5, top_k_bm25=20, top_k_context=5), rerank=RerankConfig(enabled=False, top_n=5), generator=GeneratorConfig(temperature=0.0, max_output_tokens=1024, system_prompt_path='configs/prompts/system_prompt.txt'), experiment=ExperimentConfig(limit=20, run_id=None), api=ApiConfig(gemini_api_key_env='GEMINI_API_KEY'))

## Cek Python & CUDA

In [ ]:
import torch

python_version = platform.python_version()
cuda_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_available else None

print(f"Python: {python_version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA tersedia: {cuda_available}")
print(f"GPU: {gpu_name}")

Python: 3.11.9
PyTorch: 2.6.0+cu124
CUDA tersedia: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## Verifikasi model embedding & reranker

Unduh (jika belum ada di cache HuggingFace) dan panggil sekali masing-masing model untuk
memastikan keduanya muat di VRAM dan menghasilkan output dengan bentuk yang wajar.

In [4]:
from sentence_transformers import CrossEncoder, SentenceTransformer

device = "cuda" if cuda_available else "cpu"

embedder = SentenceTransformer(cfg.models.embedding, device=device)
sample_embedding = embedder.encode(["Contoh kalimat untuk verifikasi model embedding."])
print(f"Model embedding: {cfg.models.embedding} -> dim={sample_embedding.shape[-1]}")

reranker = CrossEncoder(cfg.models.reranker, device=device)
sample_score = reranker.predict([("Contoh pertanyaan?", "Contoh konteks jawaban.")])
print(f"Model reranker: {cfg.models.reranker} -> skor contoh={sample_score}")

vram_allocated_mb = torch.cuda.memory_allocated() / 1e6 if cuda_available else None
print(f"VRAM terpakai: {vram_allocated_mb} MB" if cuda_available else "Berjalan di CPU")

d:\application\skripsi-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 19549.09it/s]


Model embedding: BAAI/bge-m3 -> dim=1024


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3332.59it/s]


Model reranker: BAAI/bge-reranker-v2-m3 -> skor contoh=[0.03181572]
VRAM terpakai: 4550.826496 MB


## Uji satu panggilan Gemini API

In [13]:
import os

from google import genai

api_key = os.environ[cfg.api.gemini_api_key_env]
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model=cfg.models.generator,
    contents="Balas hanya dengan kata 'ok'.",
)
print(f"Model generator: {cfg.models.generator} -> respons: {response.text!r}")

Model generator: gemini-3.6-flash -> respons: 'ok'


## Buat seluruh direktori proyek

In [14]:
created_dirs = ensure_directories(cfg, project_root="..")
for d in created_dirs:
    print(d)

..\data\raw\pdf
..\data\raw\web
..\data\interim
..\data\processed
..\data\benchmark
..\indexes\faiss
..\indexes\bm25
..\runs
..\results
..\figures
..\cache
..\runs\cond_a
..\runs\cond_b
..\runs\cond_c


## ASSERT

In [15]:
assert response.text and response.text.strip(), "Gemini API mengembalikan respons kosong"
assert sample_embedding.shape[-1] > 0, "Model embedding tidak menghasilkan vektor"
assert len(sample_score) == 1, "Model reranker tidak menghasilkan skor"
for d in created_dirs:
    assert Path(d).exists(), f"Direktori tidak dibuat: {d}"
print("Semua assert lolos.")

Semua assert lolos.


## Tulis `results/environment.json`

In [16]:
environment = {
    "python_version": python_version,
    "torch_version": torch.__version__,
    "cuda_available": cuda_available,
    "gpu_name": gpu_name,
    "embedding_model": cfg.models.embedding,
    "embedding_dim": int(sample_embedding.shape[-1]),
    "reranker_model": cfg.models.reranker,
    "generator_model": cfg.models.generator,
    "judge_model": cfg.models.judge,
    "seed": cfg.seed,
}

out_path = Path("../results/environment.json")
out_path.write_text(json.dumps(environment, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Ditulis ke {out_path}")

Ditulis ke ..\results\environment.json


## Ringkasan

In [17]:
print("Lingkungan siap.")
for k, v in environment.items():
    print(f"  {k}: {v}")

Lingkungan siap.
  python_version: 3.11.9
  torch_version: 2.6.0+cu124
  cuda_available: True
  gpu_name: NVIDIA GeForce RTX 4060 Laptop GPU
  embedding_model: BAAI/bge-m3
  embedding_dim: 1024
  reranker_model: BAAI/bge-reranker-v2-m3
  generator_model: gemini-3.6-flash
  judge_model: gemini-3.6-flash
  seed: 42
